# 11. Optimizer updates — exact Prodigy state chain, Muon, V4 hybrid Muon, K3 per-head Muon

Tensor shapes are small, but optimizer states and update equations are not shortened.

In [ ]:
import math

import torch

torch.manual_seed(7)
device = torch.device("cpu")
print("device:", device)

## 1. Prodigy

This is the single-parameter specialization of the reference optimizer step with `slice_p=1`, one parameter group, no FSDP and decoupled weight decay. All adaptation states (`p0`, `s`, `d_numerator`, `d_max`, Adam EMAs) and their update order are preserved.

In [ ]:
class ProdigySingleParameterState:
    def __init__(
        self,
        parameter,
        d0=1e-6,
        beta1=0.9,
        beta2=0.999,
    ):
        self.p0 = parameter.detach().clone()
        self.s = torch.zeros_like(parameter)
        self.exp_avg = torch.zeros_like(parameter)
        self.exp_avg_sq = torch.zeros_like(parameter)

        self.d0 = d0
        self.d = d0
        self.d_max = d0
        self.d_numerator = 0.0
        self.k = 0

        self.beta1 = beta1
        self.beta2 = beta2
        self.beta3 = math.sqrt(beta2)


@torch.no_grad()
def prodigy_reference_step(
    parameter,
    gradient,
    state,
    lr=1.0,
    eps=1e-8,
    weight_decay=0.01,
    d_coef=1.0,
    growth_rate=float("inf"),
    use_bias_correction=False,
    safeguard_warmup=False,
):
    beta1 = state.beta1
    beta2 = state.beta2
    beta3 = state.beta3
    d = state.d
    d0 = state.d0

    if use_bias_correction:
        bias_correction = (
            math.sqrt(1 - beta2 ** (state.k + 1))
            / (1 - beta1 ** (state.k + 1))
        )
    else:
        bias_correction = 1.0

    d_lr = d * lr * bias_correction

    state.d_numerator *= beta3
    delta_numerator = (
        (d / d0)
        * d_lr
        * torch.dot(
            gradient.flatten(),
            (state.p0 - parameter).flatten(),
        ).item()
    )

    state.exp_avg.mul_(beta1).add_(
        gradient,
        alpha=d * (1 - beta1),
    )
    state.exp_avg_sq.mul_(beta2).addcmul_(
        gradient,
        gradient,
        value=d * d * (1 - beta2),
    )

    if safeguard_warmup:
        s_alpha = (d / d0) * d
    else:
        s_alpha = (d / d0) * d_lr

    state.s.mul_(beta3).add_(gradient, alpha=s_alpha)
    d_denom = state.s.abs().sum().item()

    if d_denom > 0:
        global_numerator = state.d_numerator + delta_numerator
        d_hat = d_coef * global_numerator / d_denom

        if d == d0:
            d = max(d, d_hat)

        state.d_max = max(state.d_max, d_hat)
        d = min(state.d_max, d * growth_rate)
        state.d_numerator = global_numerator
        state.d = d

    denominator = state.exp_avg_sq.sqrt().add(d * eps)

    # Reference decoupled-weight-decay ordering.
    if weight_decay != 0:
        parameter.add_(
            parameter,
            alpha=-weight_decay * d_lr,
        )

    parameter.addcdiv_(
        state.exp_avg,
        denominator,
        value=-d_lr,
    )
    state.k += 1
    return parameter


parameter = torch.tensor(
    [[1.0, -1.0], [0.5, 2.0]],
    device=device,
)
state = ProdigySingleParameterState(parameter)

for scale in [1.0, 0.7, 0.4, 0.2]:
    gradient = scale * torch.tensor(
        [[0.2, -0.4], [1.0, 0.5]],
        device=device,
    )
    prodigy_reference_step(parameter, gradient, state)
    print("Prodigy d:", state.d)

assert state.k == 4
assert state.s.shape == parameter.shape
assert state.exp_avg.shape == parameter.shape
assert state.exp_avg_sq.shape == parameter.shape

## 2. Newton–Schulz primitive and generic Muon

In [ ]:
def newton_schulz_polynomial(x, coefficients):
    a, b, c = coefficients
    gram = x @ x.mT
    return a * x + (b * gram + c * (gram @ gram)) @ x


def normalize_matrix(matrix):
    x = matrix.float()
    transposed = x.size(-2) > x.size(-1)

    if transposed:
        x = x.mT

    x = x / (x.norm(dim=(-2, -1), keepdim=True) + 1e-7)
    return x, transposed


def nesterov_momentum(gradient, buffer, beta=0.95):
    buffer.mul_(beta).add_(gradient, alpha=1 - beta)
    return beta * buffer + (1 - beta) * gradient


def generic_muon_orthogonalize(matrix, steps=5):
    x, transposed = normalize_matrix(matrix)
    coefficients = (3.4445, -4.7750, 2.0315)

    for _ in range(steps):
        x = newton_schulz_polynomial(x, coefficients)

    if transposed:
        x = x.mT
    return x.to(matrix.dtype)


def generic_muon_direction(gradient, momentum_buffer):
    raw = nesterov_momentum(gradient, momentum_buffer)
    orthogonal = generic_muon_orthogonalize(raw)
    scale = math.sqrt(max(gradient.shape))
    return scale * orthogonal

## 3. DeepSeek-V4 hybrid Muon

The disclosed V4 matrix path uses eight aggressive Newton–Schulz iterations followed by two stable iterations, then RMS rescaling with `gamma=0.18` and decoupled weight decay. Special/non-matrix parameter groups remain on AdamW.

In [ ]:
def deepseek_v4_hybrid_orthogonalize(matrix):
    x, transposed = normalize_matrix(matrix)

    aggressive = (3.4445, -4.7750, 2.0315)
    stable = (2.0, -1.5, 0.5)

    for _ in range(8):
        x = newton_schulz_polynomial(x, aggressive)
    for _ in range(2):
        x = newton_schulz_polynomial(x, stable)

    if transposed:
        x = x.mT
    return x.to(matrix.dtype)


@torch.no_grad()
def deepseek_v4_muon_step(
    weight,
    gradient,
    momentum_buffer,
    learning_rate=0.02,
    momentum=0.95,
    weight_decay=0.1,
    gamma=0.18,
):
    raw = nesterov_momentum(
        gradient,
        momentum_buffer,
        beta=momentum,
    )
    orthogonal = deepseek_v4_hybrid_orthogonalize(raw)

    rows, columns = gradient.shape
    direction = (
        gamma
        * math.sqrt(max(rows, columns))
        * orthogonal
    )

    weight.mul_(1 - learning_rate * weight_decay)
    weight.add_(direction, alpha=-learning_rate)
    return direction


v4_weight = torch.randn(16, 12, device=device)
v4_gradient = torch.randn_like(v4_weight)
v4_momentum = torch.zeros_like(v4_weight)
v4_direction = deepseek_v4_muon_step(
    v4_weight,
    v4_gradient,
    v4_momentum,
)
print("V4 direction RMS:", v4_direction.square().mean().sqrt().item())

## 4. Kimi K3 Per-Head Muon

K3 has 96 attention heads. The example keeps all 96 head partitions and only reduces the rows per head / input width.

In [ ]:
def per_head_muon_direction(
    gradient,
    momentum_buffer,
    num_heads=96,
    beta=0.95,
):
    output_features, _ = gradient.shape
    assert output_features % num_heads == 0

    rows_per_head = output_features // num_heads
    updates = []

    for head_index in range(num_heads):
        start = head_index * rows_per_head
        stop = start + rows_per_head

        head_gradient = gradient[start:stop]
        head_momentum = momentum_buffer[start:stop]
        raw = nesterov_momentum(
            head_gradient,
            head_momentum,
            beta=beta,
        )
        orthogonal = generic_muon_orthogonalize(raw)
        scale = math.sqrt(max(head_gradient.shape))
        updates.append(scale * orthogonal)

    return torch.cat(updates, dim=0)


num_heads = 96
rows_per_head = 2
qkv_gradient = torch.randn(
    num_heads * rows_per_head,
    8,
    device=device,
)
qkv_momentum = torch.zeros_like(qkv_gradient)
per_head_update = per_head_muon_direction(
    qkv_gradient,
    qkv_momentum,
    num_heads=num_heads,
)

assert per_head_update.shape == qkv_gradient.shape
assert num_heads == 96

print("K3 head partitions:", num_heads)
print("per-head Muon update:", per_head_update.shape)

## Structural checklist

Prodigy preserves the complete D-adaptation state/update chain used by the reference implementation (special distributed and slicing options are orthogonal execution modes, not a different optimizer equation). V4 keeps the 8+2 hybrid NS chain and K3 keeps all 96 per-head partitions.